In [1]:
import cudaq
import math, random

def build_random_circuit(n_qubits: int,
                         depth: int,
                         *,
                         seed: int | None = None,
                         twoq_density: float = 0.5,       # 0..1 chance to place a 2q gate on a pair per layer
                         entangler: str = "cx",           # "cx" or "cz"
                         ring_connectivity: bool = True,  # True: (0-1,1-2,...,n-1-0), False: disjoint pairs (0-1,2-3,...)
                         measure: bool = True):
    """
    Returns a cudaq kernel implementing a random circuit with 'depth' alternating
    single-qubit and entangling layers.
    """
    rng = random.Random(seed)
    k = cudaq.make_kernel()
    q = k.qalloc(n_qubits)

    def rand_u1(target):
        r = rng.random()
        if r < 0.25:
            k.h(target)
        elif r < 0.50:
            k.rx(2 * math.pi * rng.random(), target)
        elif r < 0.75:
            k.ry(2 * math.pi * rng.random(), target)
        else:
            k.rz(2 * math.pi * rng.random(), target)

    for _ in range(depth):
        # single-qubit layer
        for i in range(n_qubits):
            rand_u1(q[i])

        # entangling layer
        if ring_connectivity:
            pairs = [(i, (i + 1) % n_qubits) for i in range(n_qubits)]
        else:
            pairs = [(i, i + 1) for i in range(0, n_qubits - 1, 2)]

        for a, b in pairs:
            if a == b:  # happens for n_qubits == 1 with ring
                continue
            if rng.random() <= twoq_density:
                if entangler == "cx":
                    k.cx(q[a], q[b])
                else:
                    k.cz(q[a], q[b])

    if measure:
        for i in range(n_qubits):
            k.mz(q[i])

    return k

# --- Example usage ---
if __name__ == "__main__":
    n, depth = 29, 25
    kernel = build_random_circuit(n, depth, seed=52, twoq_density=0.7, entangler="cz")
    # print(kernel)  # textual IR
    res = cudaq.sample(kernel, shots_count=1000)
    print(res.most_probable())  # top bitstring

00111100010110001110101010001


In [10]:
import cudaq
from typing import List, Tuple

def build_qaoa_maxcut(n_qubits: int, edges: List[Tuple[int, int]], p: int,
                      measure: bool = True):
    """
    Returns (kernel, param_count). Angles layout = [gamma_0, beta_0, ..., gamma_{p-1}, beta_{p-1}]
    """
    assert n_qubits >= 1 and p >= 1

    k, angles = cudaq.make_kernel(list[float])   # NOTE: unpack both (kernel, param_handle)
    q = k.qalloc(n_qubits)

    # |+> init
    for i in range(n_qubits):
        k.h(q[i])

    # Build layers at IR-generation time (Python loop is allowed here)
    t = 0
    for _ in range(p):
        gamma = angles[t]; t += 1
        beta  = angles[t]; t += 1

        # Cost: e^{-i γ Z_u Z_v} via CX–RZ(2γ)–CX per edge
        for (u, v) in edges:
            k.cx(q[u], q[v])
            k.rz(2.0 * gamma, q[v])
            k.cx(q[u], q[v])

        # Mixer: ∏ RX(2β)
        for i in range(n_qubits):
            k.rx(2.0 * beta, q[i])

    if measure:
        for i in range(n_qubits):
            k.mz(q[i])

    return k, 2 * p

# -------- Minimal example --------
if __name__ == "__main__":
    # Triangle graph
    E = [(0, 1), (1, 2), (2, 0)]
    qaoa, P = build_qaoa_maxcut(n_qubits=15, edges=E, p=2, measure=True)

    thetas = [0.1, 0.2, 0.1, 0.2]   # length = 2p
    shots = 2000
    res = cudaq.sample(qaoa, thetas, shots_count=shots)
    print("Most probable:", res.most_probable())

Most probable: 000100000101111


In [2]:
import cudaq
from typing import List, Tuple

def build_vqe_ansatz(n_qubits: int, layers: int, *,
                     entangler: str = "cz",
                     ring_connectivity: bool = True,
                     measure: bool = False):
    """
    Hardware-efficient ansatz built OUTSIDE the kernel.
    Params per layer per qubit: RY, RZ  =>  P = 2 * layers * n_qubits
    """
    assert n_qubits >= 1 and layers >= 1
    P = 2 * layers * n_qubits

    # Precompute connectivity (Python-side)
    if ring_connectivity:
        pairs = [(i, (i + 1) % n_qubits) for i in range(n_qubits)]
    else:
        pairs = [(i, i + 1) for i in range(0, n_qubits - 1, 2)]

    # Build kernel with a single list[float] parameter
    k, angles = cudaq.make_kernel(list[float])
    q = k.qalloc(n_qubits)

    p = 0
    for _ in range(layers):
        # Single-qubit layer
        for i in range(n_qubits):
            k.ry(angles[p], q[i]); p += 1
            k.rz(angles[p], q[i]); p += 1
        # Entangling layer
        for a, b in pairs:
            if a == b:  # avoids self-edge when n_qubits == 1
                continue
            if entangler == "cx":
                k.cx(q[a], q[b])
            else:
                k.cz(q[a], q[b])

    if measure:
        for i in range(n_qubits):
            k.mz(q[i])

    return k, P

# ---------- Minimal runnable example (energy) ----------
# H = Z0 Z1 + 0.5 (X0 + X1) using typed helpers
Z0Z1 = cudaq.spin.z(0) * cudaq.spin.z(1)
H = Z0Z1 + 0.5 * (cudaq.spin.x(0) + cudaq.spin.x(1))

ansatz, P = build_vqe_ansatz(n_qubits=2, layers=2, entangler="cz", measure=False)
thetas = [0.0] * P
obs = cudaq.observe(ansatz, H, thetas)  # expectation without in-kernel measurements
print("E =", obs.expectation())

# ---------- Sampling variant ----------
ansatz_meas, _ = build_vqe_ansatz(n_qubits=2, layers=2, entangler="cz", measure=True)
res = cudaq.sample(ansatz_meas, thetas, shots_count=1000)
print("Most probable =", res.most_probable())

E = 1.0
Most probable = 00


In [2]:
2

2